# Algorithm Scalability

This notebook showcases how `qlbm` enables the analysis of the properties of different QBM algorithms.

We consider two lenses of scalability. First, the *intrinsic* scaling of an algorithm as a function of the
lattice it models: how much deeper and wider the circuit gets as the fluid domain gains solid objects.
Second, what `qarp`'s circuit optimizer recovers from that circuit, and how long optimizing takes.

`CircuitCompiler.compile(component, optimization_level)` builds the block and, above level 0, runs `qarp`'s
peephole optimizer over it, so the sweep below varies the optimization level.

In [ ]:
import logging.config
from logging import Logger, getLogger
from typing import List

from qlbm.components import (
    CQLBM,
)
from qlbm.infra import CircuitCompiler
from qlbm.lattice import MSLattice
from qlbm.tools.utils import create_directory_and_parents, get_circuit_properties

In [ ]:
# Import analysis and plotting utilities
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme()

## Step 1: Setup

Before analyzing the scalability of the quantum circuits, we first have to choose the systems they model
and the optimization levels we compile them with. In this case, we vary the number of solid objects in
the fluid domain and study the depth and gate count of the resulting CQBM circuit, before and after
optimization.

Optimization level 0 is a no-op that only builds the block, so the sweep below uses levels 1 and 2; the
unoptimized circuit is logged separately as the baseline every algorithm is compared against.

In [ ]:
def benchmark(
    lattice_dicts,
    logger: Logger,
    dummy_logger: Logger,
    optimization_levels: List[int],
    num_repetitions: int = 5,
) -> None:
    """Compile the CQBM algorithm of each lattice at each optimization level, logging as it goes."""
    for rep in range(num_repetitions):
        logger.info(f"Repetition #{rep + 1} of {num_repetitions}")
        for count, lattice_dict in enumerate(lattice_dicts):
            for opt_count, optimization_level in enumerate(optimization_levels):
                logger.info(
                    f"Combination #{(count * len(optimization_levels)) + opt_count + 1} of {len(lattice_dicts) * len(optimization_levels)}"
                )

                lattice = MSLattice(lattice_dict, logger=dummy_logger)

                logger.info(
                    f"Lattice: {lattice.logger_name()}; opt={optimization_level}; num_qubits={lattice.num_total_qubits};"
                )

                algorithm = CQLBM(lattice)
                logger.info(
                    f"Original circuit has properties: {get_circuit_properties(algorithm)}"
                )

                compiler = CircuitCompiler(logger=logger)
                compiler.compile(
                    algorithm,
                    optimization_level=optimization_level,
                )

In [ ]:
ROOT_OUTPUT_DIR = "qlbm-output/benchmark-algorithm-scalability"
LOG_FILE = f"{ROOT_OUTPUT_DIR}/qlbm.log"

create_directory_and_parents(ROOT_OUTPUT_DIR)

# Truncate the log so that re-running the notebook does not append to a previous session
with open(LOG_FILE, "w"):
    pass

In [ ]:
lattices = [
    {
        "lattice": {"dim": {"x": 8, "y": 8}, "velocities": {"x": 4, "y": 4}},
        "geometry": [],
    },  # 0 Obstacles
    {
        "lattice": {"dim": {"x": 8, "y": 8}, "velocities": {"x": 4, "y": 4}},
        "geometry": [
            {"shape": "cuboid", "x": [5, 6], "y": [1, 2], "boundary": "specular"}
        ],
    },  # 1 Obstacle
    {
        "lattice": {"dim": {"x": 8, "y": 8}, "velocities": {"x": 4, "y": 4}},
        "geometry": [
            {"shape": "cuboid", "x": [5, 6], "y": [1, 2], "boundary": "specular"},
            {"shape": "cuboid", "x": [5, 6], "y": [5, 6], "boundary": "specular"},
        ],
    },  # 2 Obstacles
]

OPTIMIZATION_LEVELS = [1, 2]
NUM_REPS = 3

dummy_logger = getLogger("dummy")
# By logging at this point we ignore the output of circuit creation
logging.config.fileConfig("algorithm_scalability_logging.conf")
logger = getLogger("qlbm")

## Step 2: Benchmarking

Now that we have set up our sweep and configured logging, we can run it by calling the `benchmark`
function. The sweep below is demo-sized (3 lattices, 2 optimization levels, 3 repetitions) and takes
well under a minute; widen `lattices`, `OPTIMIZATION_LEVELS`, or `NUM_REPS` for a real study.

> **_CAUTION:_** The cell below should be run exactly once per session, since it appends to the log
> that Step 3 parses.

In [ ]:
logger.info("Session: QARP")
benchmark(
    lattices,
    logger,
    dummy_logger,
    OPTIMIZATION_LEVELS,
    num_repetitions=NUM_REPS,
)

## Step 3: Analysis

Once the compilation benchmarks have concluded, the performance logs created by `qlbm` will be under
`qlbm-output/benchmark-algorithm-scalability/qlbm.log`.
The script below parses that file, extracts the circuit properties `qlbm` logged, formats them, and
plots them for convenient analysis.

In [ ]:
with open(LOG_FILE, "r") as f:
    lines = f.readlines()

combination_line_indices = [
    c for c, line in enumerate(lines) if "Combination #" in line
]
sections = [
    lines[start:end]
    for start, end in zip(
        combination_line_indices, combination_line_indices[1:] + [len(lines)]
    )
]

records = []
for section in sections:
    section_info = (
        [line for line in section if "Lattice: " in line][0]
        .rstrip("\n")
        .split(": ")[-1]
        .split("; ")
    )
    lattice_name = section_info[0]
    opt_level = int(section_info[1].split("=")[-1])
    num_qubits = int(section_info[2].split("=")[-1][:-1])

    original_props = (
        [line for line in section if "Original circuit" in line][0]
        .split("(")[-1]
        .rstrip("\n)")
        .split(", ")
    )
    compiled_props = (
        [line for line in section if "Compiled block has properties" in line][0]
        .split("(")[-1]
        .rstrip("\n)")
        .split(", ")
    )
    duration = int(
        [line for line in section if "Compilation took" in line][0].split()[-2]
    )

    records.append(
        {
            "Lattice": lattice_name,
            "Dimensions": lattice_name.split("-")[1],
            "Obstacles": int(lattice_name.split("-")[2]),
            "Circuit Qubits": num_qubits,
            "Optimization Level": opt_level,
            "Initial Depth": int(original_props[2]),
            "Initial Gate No.": int(original_props[3]),
            "Compiled Depth": int(compiled_props[2]),
            "Compiled Gate No.": int(compiled_props[3]),
            "Duration (ns)": duration,
            "Duration (s)": duration / 1e9,
        }
    )

df = pd.DataFrame.from_records(records)

In [ ]:
# We can have a look at the raw data
df

In [ ]:
sns.lineplot(
    df,
    x="Obstacles",
    y="Initial Depth",
    markers=True,
)
plt.xticks(pd.unique(df["Obstacles"]))
plt.title("Circuit Depth Scaling")

In [ ]:
sns.lineplot(
    df,
    x="Obstacles",
    y="Initial Gate No.",
    markers=True,
)
plt.xticks(pd.unique(df["Obstacles"]))
plt.title("Circuit Gate Number Scaling")

In [ ]:
sns.lineplot(
    df,
    x="Obstacles",
    y="Compiled Depth",
    hue="Optimization Level",
    style="Optimization Level",
    markers=True,
)
plt.xticks(pd.unique(df["Obstacles"]))
plt.title("Optimized Depth Comparison")

In [ ]:
sns.lineplot(
    df,
    x="Obstacles",
    y="Compiled Gate No.",
    hue="Optimization Level",
    style="Optimization Level",
    markers=True,
)
plt.xticks(pd.unique(df["Obstacles"]))
plt.title("Optimized Gate Comparison")

In [ ]:
g = sns.lineplot(
    df,
    x="Obstacles",
    y="Duration (s)",
    hue="Optimization Level",
    style="Optimization Level",
    markers=True,
)
plt.xticks(pd.unique(df["Obstacles"]))
plt.title("Optimization Duration Comparison")